# Live Tracking

Loads a model and uses its predictions on a live camera feed (e.g. the webcam)

## 1. Dependencies

This cell installs and checks if all necessary dependencies are installed.
If not execute in the root directory ``pip install -r requirments.txt``

In [ ]:
import importlib.util

REQUIRED = ["ultralytics", "lap"]

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "torch is not installed. Install it from https://pytorch.org/get-started/locally/ "
        "with the index URL matching your CUDA version"
    )

missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print("Missing:", " ".join(missing))
else:
    print("All dependencies installed.")

import torch
import ultralytics

print(f"torch {torch.__version__}   ultralytics {ultralytics.__version__}")
print(f"cuda {torch.version.cuda or 'n/a'}   available {torch.cuda.is_available()}")

## 2. Setup

Import the Ultralytics adapter from the source.

In [ ]:
import sys
from pathlib import Path


def repo_root(start: Path | None = None) -> Path:
    """Walk up from the notebook until the directory holding the model package appears."""
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "src" / "model.py").exists() or (cand / "model.py").exists():
            return cand
    raise FileNotFoundError(f"no repo root above {here}")


ROOT = repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))

print("repo root:", ROOT)


def import_adapter():
    """Import the Ultralytics adapter."""
    import importlib

    tried = []
    for name in ("src.ultralytics_adapter", "ultralytics_adapter", f"{ROOT.name}.ultralytics_adapter"):
        try:
            return importlib.import_module(name)
        except ImportError as exc:
            tried.append(f"  {name:40s} -> {exc}")
    raise ImportError(
        "could not import ultralytics_adapter. Tried:\n" + "\n".join(tried) +
        "\n\nThe adapter defines MyDetectionModel/MyTrainer/MyYOLO sit"
        "\nnext to model.py. This checkout only has its compiled __pycache__ copy."
    )


adapter = import_adapter()
print("adapter:", adapter.__name__)

from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "cpu")

## 3. Select the model

Change the CKPT variable if you want to use another model then our pre-trained one.

In [ ]:
CKPT = ROOT / "pre_trained" / "my_yolo26_n.pt"

model = YOLO(CKPT)
print(CKPT.name, "--", model.model.names)

## 4. Track

Adjust the source value to select another camera if ``0`` does not work.

In [ ]:
for _ in model.track(source=2, show=True, stream=True,
                     tracker="bytetrack.yaml", conf=0.35, imgsz=640, verbose=False):
    pass